# MDP Practice — Value Iteration
**Practice date:** 2026-07-01  
**Topic:** Markov Decision Processes (MDP) + Dynamic Programming

This is a **practice template**. Some code is filled in for you; the parts you need to write are marked with `# TODO`. Work through it top to bottom.

---

## The problem: Frozen Lake (deterministic version)

A frozen lake is a 4×4 grid. You start at the top-left tile `S` and want to reach the goal `G` at the bottom-right, **without falling into a hole `H`**.

```
S F F F
F H F H
F F F H
H F F G
```

- `S` = start,  `F` = frozen (safe),  `H` = hole (you fall, episode ends),  `G` = goal.
- Reaching `G` gives reward **+1**. Everything else gives **0**.
- We use the **non-slippery** version, so an action always moves you the way you intend (this makes the MDP easy to reason about while you learn).

## Recap: what is an MDP?
An MDP is 5 pieces — **(S, A, P, R, γ)**:

| Symbol | Name | In Frozen Lake |
|--------|------|----------------|
| **S** | States | the 16 grid tiles (0–15) |
| **A** | Actions | 0=Left, 1=Down, 2=Right, 3=Up |
| **P** | Transitions | `env.P[s][a]` = where you land |
| **R** | Reward | +1 at the goal, else 0 |
| **γ** (gamma) | Discount | how much we value future reward (we'll use 0.9) |

## Your goal today
Use **value iteration** (a Dynamic Programming method) to compute:
1. `V(s)` — how good each tile is, and
2. the **best action** to take on each tile (the optimal policy).

Let's go. ⬇️

## Step 0 — Setup (already done for you)
Just run this cell. If you get a `ModuleNotFoundError`, run the `!pip install gymnasium` cell below it first.

In [ ]:
import gymnasium as gym
import numpy as np

# Deterministic Frozen Lake: is_slippery=False means an action ALWAYS moves you the way you intend.
env = gym.make('FrozenLake-v1', is_slippery=False)
env.reset()

n_states  = env.observation_space.n   # 16 tiles
n_actions = env.action_space.n        # 4 moves
print('Number of states :', n_states)
print('Number of actions:', n_actions, ' (0=Left, 1=Down, 2=Right, 3=Up)')

In [ ]:
# Only run this if the import above failed.
!pip install gymnasium

## Step 1 — Look at the MDP model `env.P`
`env.unwrapped.P[s][a]` gives a **list of outcomes**, where each outcome is a tuple:

```
(probability, next_state, reward, done)
```

Run the cell and read the output — this is the entire "rulebook" of the world. Value iteration will read exactly these numbers.

In [ ]:
# The 4 possible moves from the START tile (state 0)
for a in range(n_actions):
    print(f'Action {a}: {env.unwrapped.P[0][a]}')

# The tile just left of the goal (state 14): taking action Right (2) should reach the goal (15)
print('\nFrom state 14, action Right (2):', env.unwrapped.P[14][2])

## Step 2 — Write `value_iteration`  ✏️ (YOUR TASK)

The idea: repeatedly update every state's value using the **Bellman optimality equation**:

$$V(s) \leftarrow \max_{a} \sum_{\text{outcomes}} p \cdot \big(r + \gamma \, V(s')\big)$$

In words: *"The value of a tile = the value of its BEST action. An action's value = for each place it might send you, take (reward + discounted value of that place), weighted by how likely it is."*

Fill in the two `# TODO` spots. Everything else is done.

In [ ]:
def value_iteration(env, gamma=0.9, threshold=1e-8):
    n_states  = env.observation_space.n
    n_actions = env.action_space.n
    value_table = np.zeros(n_states)   # start every tile's value at 0

    while True:
        old_value_table = np.copy(value_table)

        for s in range(n_states):
            Q_values = []                       # one Q-value per action
            for a in range(n_actions):
                q = 0
                for prob, next_state, reward, done in env.unwrapped.P[s][a]:
                    # TODO 1: add this outcome's contribution to q.
                    #   formula:  prob * (reward + gamma * value_table[next_state])
                    #   (use old_value_table or value_table — both work here)
                    q += 0   # <-- REPLACE the 0 with the formula above
                Q_values.append(q)

            # TODO 2: the new value of state s is the value of its BEST action.
            value_table[s] = 0   # <-- REPLACE 0 with the best (maximum) of Q_values

        # stop once the values barely change between sweeps
        if np.sum(np.abs(old_value_table - value_table)) <= threshold:
            break

    return value_table

## Step 3 — Write `extract_policy`  ✏️ (YOUR TASK)

Now that we know how good each tile is, the **best action** on tile `s` is the one with the highest Q-value. This is almost identical to Step 2, except we keep the *action number* (argmax) instead of the *value* (max).

Fill in the two `# TODO` spots.

In [ ]:
def extract_policy(env, value_table, gamma=0.9):
    n_states  = env.observation_space.n
    n_actions = env.action_space.n
    policy = np.zeros(n_states)

    for s in range(n_states):
        Q_values = []
        for a in range(n_actions):
            q = 0
            for prob, next_state, reward, done in env.unwrapped.P[s][a]:
                # TODO 1: same formula as before
                q += 0   # <-- REPLACE with prob * (reward + gamma * value_table[next_state])
            Q_values.append(q)

        # TODO 2: store the BEST ACTION number for state s.
        policy[s] = 0   # <-- REPLACE 0 with np.argmax(Q_values)

    return policy

## Step 4 — Run it and see your answer

In [ ]:
V = value_iteration(env)
print('Value of each tile (higher = closer/safer path to goal):')
print(V.reshape(4, 4).round(3))

policy = extract_policy(env, V)
print('\nBest action on each tile  (0=Left, 1=Down, 2=Right, 3=Up):')
print(policy.reshape(4, 4).astype(int))

## Step 5 — How to check you got it right ✅

If your code is correct, you should see:

- **Values** that get **larger as tiles get closer to the goal** (bottom-right). Hole tiles and the goal stay at 0.
- A **policy that points toward the goal** — following the arrows from the start (top-left) should walk you around the holes down to `G`, never stepping into an `H`.

If everything is 0, you probably left a `# TODO` unfilled. 🙂

### Stretch goals (optional)
1. Change `gamma` to `1.0` and then `0.5`. How does the value table change? Why?
2. Switch to the **slippery** lake (`is_slippery=True`) and re-run. Does the best policy change? (It should — the agent now has to play it safe around holes.)
3. Print the policy using arrows (`←`, `↓`, `→`, `↑`) instead of numbers to make it easier to read.